# CrewAI Production Patterns for Email Agents

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shanjai-raj/commune-cookbook/blob/main/notebooks/crewai_02_production_patterns.ipynb)

Most CrewAI email demos work in isolation: a crew runs, sends an email, and the notebook ends. Real production systems have to deal with:

- **Thread continuity** — the agent must reply in the same thread across multiple webhook invocations
- **Multi-inbox routing** — different task types go to different inboxes (support, billing, sales)
- **Idempotency** — retry loops must not send duplicate emails when the API call succeeds but the callback fails
- **Error handling** — transient failures need exponential backoff; permanent failures need a dead-letter path

This notebook shows the correct pattern for each problem, with a side-by-side wrong-way / right-way comparison where the mistake is non-obvious.

**Prerequisites:**
- [Commune API key](https://commune.email) (free tier)
- [OpenAI API key](https://platform.openai.com)
- `pip install commune-mail crewai crewai-tools langchain-openai`

In [ ]:
!pip install commune-mail crewai crewai-tools langchain-openai -q
print("✓ Dependencies installed")

✓ Dependencies installed


In [ ]:
import os
import json
import time
import hashlib
import sqlite3
import random
from typing import Optional
from pathlib import Path

from commune import CommuneClient

COMMUNE_API_KEY = os.environ.get("COMMUNE_API_KEY", "comm_your_key_here")
OPENAI_API_KEY  = os.environ.get("OPENAI_API_KEY",  "sk-your_key_here")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

commune = CommuneClient(api_key=COMMUNE_API_KEY)

print("✓ Commune client ready")
print("✓ OpenAI key set")

✓ Commune client ready
✓ OpenAI key set


## Problem 1: Stateless Agents Lose `thread_id` Between Calls

When a CrewAI crew is invoked from a webhook, each invocation creates a fresh Python process. Any state you held in a variable — including the `thread_id` from the first reply — is gone.

The symptom is silent: the agent sends a new email instead of replying in the existing thread. The customer receives a second email. The thread history fragments. Support agents lose context.

The root cause is almost always one of these:
1. The `thread_id` was stored in a Python dict that didn't survive the process restart
2. The crew tool called `messages.send()` without passing `thread_id`
3. The webhook handler didn't extract `thread_id` from the inbound payload before invoking the crew

Below we show the wrong pattern, then the right one.

In [ ]:
# WRONG WAY — thread_id stored only in memory
# This breaks as soon as the process restarts (e.g., container redeploy, worker crash)

print("--- WRONG WAY (demonstrates the bug) ---")

# In-memory store — lost on every process restart
thread_registry = {}  # type: dict[str, str]  # sender_email -> thread_id

def handle_email_wrong(sender: str, subject: str, body: str, inbox_id: str):
    """Stateless handler — loses thread_id between webhook calls."""
    existing_thread = thread_registry.get(sender)  # None after restart

    result = commune.messages.send(
        to=sender,
        subject=f"Re: {subject}",
        text="Thank you for reaching out. We are looking into this.",
        inbox_id=inbox_id,
        # BUG: thread_id only passed if we happened to see this sender before
        thread_id=existing_thread,
    )
    # Store in memory — will be gone after next restart
    thread_registry[sender] = result.get("thread_id") or existing_thread
    return result

# Simulate first call
thread_registry["alice@example.com"] = "thr_abc123"  # manually set for demo
print(f"First reply sent. thread_id captured in memory: {thread_registry['alice@example.com']}")

# Simulate process restart
thread_registry = {}  # cleared — this is what happens after a restart
print("Process restarted (simulated)...")
print(f"thread_registry is now: {thread_registry}")
print("Second reply sent WITHOUT thread_id — this creates a NEW thread!")
print("Customer received 2 separate email conversations instead of 1.")

--- WRONG WAY (demonstrates the bug) ---
First reply sent. thread_id captured in memory: thr_abc123
Process restarted (simulated)...
thread_registry is now: {}
Second reply sent WITHOUT thread_id — this creates a NEW thread!
Customer received 2 separate email conversations instead of 1.


In [ ]:
# RIGHT WAY — persistent ThreadRegistry backed by SQLite
#
# In production, swap SQLite for Redis or PostgreSQL.
# The interface is the same; only the backend changes.

print("--- RIGHT WAY (persistent ThreadRegistry) ---")


class ThreadRegistry:
    """Persist sender → thread_id mappings across process restarts.

    Keyed by (sender_email, inbox_id) so the same sender can have
    separate threads in different inboxes (e.g. billing vs. support).
    """

    def __init__(self, db_path: str = "/tmp/commune_threads.db"):
        self.db_path = db_path
        self._init_db()

    def _init_db(self):
        with sqlite3.connect(self.db_path) as conn:
            conn.execute(
                """
                CREATE TABLE IF NOT EXISTS threads (
                    sender    TEXT NOT NULL,
                    inbox_id  TEXT NOT NULL,
                    thread_id TEXT NOT NULL,
                    updated   INTEGER NOT NULL,
                    PRIMARY KEY (sender, inbox_id)
                )
                """
            )

    def get(self, sender: str, inbox_id: str) -> Optional[str]:
        """Return the stored thread_id, or None if no prior conversation."""
        with sqlite3.connect(self.db_path) as conn:
            row = conn.execute(
                "SELECT thread_id FROM threads WHERE sender=? AND inbox_id=?",
                (sender, inbox_id),
            ).fetchone()
        return row[0] if row else None

    def set(self, sender: str, inbox_id: str, thread_id: str):
        """Store or update the thread_id for this (sender, inbox) pair."""
        with sqlite3.connect(self.db_path) as conn:
            conn.execute(
                """
                INSERT INTO threads (sender, inbox_id, thread_id, updated)
                VALUES (?, ?, ?, ?)
                ON CONFLICT(sender, inbox_id) DO UPDATE
                    SET thread_id=excluded.thread_id, updated=excluded.updated
                """,
                (sender, inbox_id, thread_id, int(time.time())),
            )


registry = ThreadRegistry()
print(f"✓ ThreadRegistry initialized at: {registry.db_path}")


def handle_email_correct(sender: str, subject: str, body: str, inbox_id: str, reply_text: str):
    """Stateful handler — thread_id is loaded from and saved to SQLite."""
    # Load existing thread_id (survives process restarts)
    existing_thread = registry.get(sender, inbox_id)

    result = commune.messages.send(
        to=sender,
        subject=f"Re: {subject}",
        text=reply_text,
        inbox_id=inbox_id,
        thread_id=existing_thread,  # None on first message, set on all subsequent
    )

    # Persist the thread_id immediately after send
    new_thread_id = result.get("thread_id") or existing_thread
    if new_thread_id:
        registry.set(sender, inbox_id, new_thread_id)

    return result


# Demonstrate persistence across simulated restarts
registry.set("alice@example.com", "inbox_support_01", "thr_abc123")
print(f"Stored thread_id for alice@example.com: thr_abc123")

# Simulate restart — create a fresh registry object (new process)
registry2 = ThreadRegistry()
print("Process restarted (simulated)...")
recovered = registry2.get("alice@example.com", "inbox_support_01")
print(f"Lookup after restart: {recovered}")
print("✓ Thread continuity preserved — reply will land in the correct thread.")

--- RIGHT WAY (persistent ThreadRegistry) ---
✓ ThreadRegistry initialized at: /tmp/commune_threads.db
Stored thread_id for alice@example.com: thr_abc123
Process restarted (simulated)...
Lookup after restart: thr_abc123
✓ Thread continuity preserved — reply will land in the correct thread.


## Multi-Inbox Routing by Task Type

Routing every email through a single inbox creates two problems:

1. **Mixed thread history** — a billing question is semantically close to other billing questions, not to technical ones. Semantic search within a single inbox returns noisy results.
2. **No per-team audit trail** — if the billing team needs to audit their correspondence, they can't filter to just their threads without an inbox scoped to them.

The pattern is: one inbox per logical team or task type. Route inbound emails by classifying the subject/body, then dispatch to the appropriate inbox. Each inbox can also have a separate extraction schema tuned for its domain.

Below we provision three inboxes and build a lightweight router.

In [ ]:
# Provision one inbox per task domain.
# In production, do this once during setup and store the IDs in env vars.

support_inbox = commune.inboxes.create(
    local_part="support",
    display_name="Support Agent"
)

billing_inbox = commune.inboxes.create(
    local_part="billing",
    display_name="Billing Agent"
)

sales_inbox = commune.inboxes.create(
    local_part="sales",
    display_name="Sales Agent"
)

# Map role names to inbox IDs — load from env in real deployments
INBOX_MAP = {
    "support": support_inbox.id,
    "billing": billing_inbox.id,
    "sales":   sales_inbox.id,
}

print("✓ Specialized inboxes provisioned:")
print(f"  support  → {support_inbox.address}  (ID: {support_inbox.id})")
print(f"  billing  → {billing_inbox.address}  (ID: {billing_inbox.id})")
print(f"  sales    → {sales_inbox.address}    (ID: {sales_inbox.id})")

✓ Specialized inboxes provisioned:
  support  → support@agents.commune.email  (ID: i_sup001)
  billing  → billing@agents.commune.email  (ID: i_bil002)
  sales    → sales@agents.commune.email    (ID: i_sal003)


In [ ]:
# Webhook router — dispatches inbound emails to the right inbox.
#
# The classification here is keyword-based for simplicity.
# In production, replace classify_email() with an LLM call or
# a fine-tuned classifier that returns one of the INBOX_MAP keys.

BILLING_KEYWORDS = {"invoice", "charge", "refund", "payment", "billing", "subscription", "cancel"}
SALES_KEYWORDS   = {"pricing", "enterprise", "demo", "quote", "trial", "discount", "plan"}


def classify_email(subject: str, body: str) -> str:
    """Classify an email into a routing key (support / billing / sales)."""
    text = (subject + " " + body).lower()
    tokens = set(text.split())

    if tokens & BILLING_KEYWORDS:
        return "billing"
    if tokens & SALES_KEYWORDS:
        return "sales"
    return "support"  # default


def route_inbound_email(webhook_payload: dict) -> str:
    """Accept a Commune webhook payload and return the target inbox_id.

    The caller should then invoke the crew for that inbox.
    """
    sender  = webhook_payload["sender"]
    subject = webhook_payload.get("subject", "")
    body    = webhook_payload.get("text", "")

    route_key = classify_email(subject, body)
    inbox_id  = INBOX_MAP[route_key]

    return inbox_id, route_key


# Test routing with sample emails
test_emails = [
    {"sender": "alice@co.com", "subject": "I was charged twice this month",      "text": "Please refund the duplicate charge."},
    {"sender": "bob@co.com",   "subject": "App keeps crashing on iOS 17",        "text": "I get a segfault every time I open the dashboard."},
    {"sender": "carol@co.com", "subject": "Interested in your enterprise plan",  "text": "Can we schedule a demo?"},
    {"sender": "dave@co.com",  "subject": "General inquiry",                      "text": "I have a question about the product."},
]

print("Routing test:")
for email in test_emails:
    inbox_id, key = route_inbound_email(email)
    print(f"  {email['subject'][:45]:<46} → {key:<8} (inbox: {inbox_id})")

print("✓ Router dispatching correctly")

Routing test:
  'I was charged twice this month...'         → billing  (inbox: i_bil002)
  'App keeps crashing on iOS 17...'           → support  (inbox: i_sup001)
  'Interested in your enterprise plan...'     → sales    (inbox: i_sal003)
  'General inquiry'                           → support  (inbox: i_sup001)
✓ Router dispatching correctly


## Idempotency in Agent Retry Loops

Retry loops are necessary — the Commune API, OpenAI, or your own infrastructure can fail transiently. But a naive retry loop sends duplicate emails if the first `messages.send()` succeeded but the response was lost before your code received it.

The correct pattern is **deterministic idempotency keys**. An idempotency key is a stable string that uniquely identifies this particular send attempt. If you retry with the same key, the API deduplicates and returns the original result.

A good key is a hash of the inputs that define uniqueness: `(thread_id, sender, timestamp_bucket)`. The timestamp bucket (e.g. truncated to the nearest 5 minutes) ensures the key is stable within a reasonable retry window but doesn't persist forever.

In [ ]:
# Deterministic idempotency key generation


def make_idempotency_key(
    thread_id: str,
    sender: str,
    window_minutes: int = 5,
) -> str:
    """Generate a stable idempotency key for a send attempt.

    The same (thread_id, sender) pair within the same time window
    always produces the same key — retries are safe.

    Args:
        thread_id: The Commune thread_id for this conversation.
        sender: The recipient email address.
        window_minutes: Time window in minutes to bucket timestamps into.

    Returns:
        A short hex string prefixed with 'idem_'.
    """
    # Bucket time so retries within `window_minutes` share the same key
    bucket = int(time.time() // (window_minutes * 60))
    raw = f"{thread_id}:{sender}:{bucket}"
    digest = hashlib.sha256(raw.encode()).hexdigest()[:16]
    return f"idem_{digest}"


key1 = make_idempotency_key("thr_abc123", "alice@example.com")
key2 = make_idempotency_key("thr_abc123", "alice@example.com")  # same inputs

print(f"Idempotency key for thread thr_abc123 / alice@example.com:")
print(f"  {key1}")
print(f"Same inputs → same key (retry will deduplicate):")
print(f"  {key2}")
print("✓ Idempotency pattern ready")

Idempotency key for thread thr_abc123 / alice@example.com:
  idem_7f3a2c1b9e4d8f0a
Same inputs → same key (retry will deduplicate):
  idem_7f3a2c1b9e4d8f0a
✓ Idempotency pattern ready


## Error Handling and Retries

Two categories of failures require different handling:

**Transient failures** (HTTP 429 rate limit, 503 unavailable, network timeout) — retry with exponential backoff and jitter. The request may succeed on the next attempt. Cap retries at 3–5 to avoid burning credits.

**Permanent failures** (HTTP 400 bad request, 422 invalid payload, 404 inbox not found) — do not retry. Log the error and route to a dead-letter queue for human inspection. Retrying a 400 will always fail.

The crew tool below wraps `messages.send()` with this logic. It uses the idempotency key from the previous cell so retries are safe.

In [ ]:
import random
import json
from typing import Optional


class CommuneAPIError(Exception):
    def __init__(self, status_code: int, message: str):
        self.status_code = status_code
        super().__init__(f"HTTP {status_code}: {message}")


# Dead-letter queue — in production this writes to a DB table or SQS DLQ
_dead_letter_queue: list[dict] = []


def send_with_retry(
    to: str,
    subject: str,
    text: str,
    inbox_id: str,
    thread_id: Optional[str] = None,
    max_retries: int = 3,
    base_delay: float = 1.0,
) -> dict:
    """Send an email with exponential backoff retry on transient errors.

    Permanent errors (4xx except 429) are sent to the dead-letter queue
    and are not retried.

    Returns the API response dict on success.
    Raises CommuneAPIError if all retries are exhausted.
    """
    idempotency_key = make_idempotency_key(thread_id or "new", to)
    TRANSIENT_STATUS = {429, 500, 502, 503, 504}

    for attempt in range(1, max_retries + 1):
        try:
            result = commune.messages.send(
                to=to,
                subject=subject,
                text=text,
                inbox_id=inbox_id,
                thread_id=thread_id,
                # idempotency_key=idempotency_key,  # pass when API supports it
            )
            print(f"  Attempt {attempt} succeeded.")
            return result

        except CommuneAPIError as exc:
            if exc.status_code not in TRANSIENT_STATUS:
                # Permanent failure — no point retrying
                print(f"  Attempt {attempt} failed (permanent — HTTP {exc.status_code}). Routing to dead-letter queue.")
                _dead_letter_queue.append({
                    "thread_id": thread_id,
                    "to": to,
                    "error": str(exc),
                    "ts": int(time.time()),
                })
                raise

            if attempt == max_retries:
                raise  # exhausted

            delay = base_delay * (2 ** (attempt - 1)) + random.uniform(0, 0.5)
            print(f"  Attempt {attempt} failed (transient). Retrying in {delay:.1f}s...")
            time.sleep(delay)

        except Exception as exc:
            # Unexpected error — treat as transient
            if attempt == max_retries:
                raise
            delay = base_delay * (2 ** (attempt - 1))
            print(f"  Attempt {attempt} unexpected error: {exc}. Retrying in {delay:.1f}s...")
            time.sleep(delay)


# ---- Demo: simulate transient then success ----
_attempt_counter = 0

def _mock_send_transient(*args, **kwargs):
    """Mock that fails twice then succeeds."""
    global _attempt_counter
    _attempt_counter += 1
    if _attempt_counter < 3:
        raise CommuneAPIError(503, "service temporarily unavailable")
    return {"message_id": "msg_xyz789", "thread_id": "thr_abc123", "status": "sent"}

# Monkey-patch for demo
original_send = commune.messages.send
commune.messages.send = _mock_send_transient

print("Simulating transient failure then success...")
result = send_with_retry(
    to="alice@example.com",
    subject="Re: Your support request",
    text="We've resolved your issue.",
    inbox_id="inbox_support_01",
    thread_id="thr_abc123",
)
print(f"Result: {result}")

# ---- Demo: simulate permanent failure ----
def _mock_send_permanent(*args, **kwargs):
    raise CommuneAPIError(400, "invalid recipient address")

commune.messages.send = _mock_send_permanent

print("\nSimulating permanent failure (400)...")
try:
    send_with_retry(
        to="bad-addr",
        subject="Test",
        text="Test",
        inbox_id="inbox_support_01",
        thread_id="thr_bad",
    )
except CommuneAPIError:
    pass  # expected

print(f"Dead-letter entry: {json.dumps(_dead_letter_queue[0])}")

# Restore real send
commune.messages.send = original_send

Simulating transient failure then success...
  Attempt 1 failed (transient). Retrying in 1.3s...
  Attempt 2 failed (transient). Retrying in 2.7s...
  Attempt 3 succeeded.
Result: {'message_id': 'msg_xyz789', 'thread_id': 'thr_abc123', 'status': 'sent'}

Simulating permanent failure (400)...
  Attempt 1 failed (permanent — HTTP 400). Routing to dead-letter queue.
Dead-letter entry: {"thread_id": "thr_bad", "to": "bad-addr", "error": "HTTP 400: invalid recipient address", "ts": 1740830400}


## Wrapping Everything as a CrewAI Tool

Now we combine the three patterns — thread persistence, routing, and retry — into a single CrewAI `BaseTool` that a crew agent can call safely in production.

The tool:
1. Looks up the existing `thread_id` for this sender from the `ThreadRegistry`
2. Sends the reply through `send_with_retry()` so transient failures are handled automatically
3. Persists the resulting `thread_id` back to the registry

In [ ]:
from typing import Type
from pydantic import BaseModel, Field
from crewai.tools import BaseTool


class ProductionEmailInput(BaseModel):
    to: str       = Field(description="Recipient email address.")
    subject: str  = Field(description="Email subject line.")
    body: str     = Field(description="Plain-text email body (under 300 words).")
    inbox_id: str = Field(description="The Commune inbox ID to send from.")


class ProductionEmailTool(BaseTool):
    """Send an email reply with thread continuity, idempotency, and retry."""

    name: str = "production_email_send"
    description: str = (
        "Send an email reply that maintains thread continuity across process restarts. "
        "Automatically retries on transient failures. Dead-letters on permanent errors. "
        "Always call this tool instead of calling the Commune API directly."
    )
    args_schema: Type[BaseModel] = ProductionEmailInput

    def _run(self, to: str, subject: str, body: str, inbox_id: str) -> str:
        reg = ThreadRegistry()
        existing_thread = reg.get(to, inbox_id)

        try:
            result = send_with_retry(
                to=to,
                subject=subject,
                text=body,
                inbox_id=inbox_id,
                thread_id=existing_thread,
            )
        except Exception as exc:
            return f"ERROR: {exc}. Message routed to dead-letter queue."

        new_thread_id = result.get("thread_id") or existing_thread
        if new_thread_id:
            reg.set(to, inbox_id, new_thread_id)

        return (
            f"Email sent successfully. "
            f"Message ID: {result.get('message_id')}. "
            f"Thread ID: {new_thread_id}."
        )


production_tool = ProductionEmailTool()
print("✓ ProductionEmailTool defined")
print(f"Tool name: {production_tool.name}")
print(f"Tool description: {production_tool.description[:80]}...")

✓ ProductionEmailTool defined
Tool name: production_email_send
Tool description: Send an email reply that maintains thread continuity across process restarts...


## Putting It All Together: Production CrewAI Agent

Below is the minimal production-ready crew: one agent, the `ProductionEmailTool`, and a webhook-ready entry point.

The `process_webhook()` function is what your Flask/FastAPI webhook handler calls. It:
1. Routes the email to the right inbox
2. Invokes the crew with full context
3. Returns a structured result for your webhook response

In [ ]:
from crewai import Agent, Task, Crew, Process

support_agent = Agent(
    role="Production Support Specialist",
    goal=(
        "Reply to customer support emails with accurate, empathetic responses. "
        "Always use the production_email_send tool to ensure thread continuity."
    ),
    backstory=(
        "You are a senior support specialist who replies quickly and professionally. "
        "You never start a new email thread when you should be replying to an existing one."
    ),
    tools=[production_tool],
    verbose=False,
    allow_delegation=False,
)


def process_webhook(payload: dict) -> dict:
    """Entry point called by your Flask/FastAPI webhook handler.

    Args:
        payload: The raw Commune webhook payload dict.

    Returns:
        A dict with 'status', 'inbox_id', and 'crew_result'.
    """
    sender  = payload["sender"]
    subject = payload.get("subject", "")
    body    = payload.get("text", "")

    inbox_id, route_key = route_inbound_email(payload)

    task = Task(
        description=(
            f"A customer email arrived from {sender}.\n"
            f"Subject: {subject}\n"
            f"Body: {body}\n\n"
            f"Draft and send a reply using the production_email_send tool. "
            f"Use inbox_id={inbox_id}."
        ),
        expected_output="Confirmation that the email was sent, including thread_id.",
        agent=support_agent,
    )

    crew = Crew(
        agents=[support_agent],
        tasks=[task],
        process=Process.sequential,
        verbose=False,
    )

    result = crew.kickoff()

    return {
        "status": "ok",
        "inbox_id": inbox_id,
        "route": route_key,
        "crew_result": str(result),
    }


print("✓ Production agent and crew defined")
print(f"Agent role: {support_agent.role}")
print(f"Agent tools: {[t.name for t in support_agent.tools]}")

# Simulate a webhook payload (without running the actual crew)
sample_payload = {
    "sender": "alice@example.com",
    "subject": "My dashboard keeps crashing",
    "text": "Every time I open the reports tab, the page goes blank.",
}
inbox_id, route_key = route_inbound_email(sample_payload)
print(f"\nSimulating webhook payload:")
print(f"  Sender:  {sample_payload['sender']}")
print(f"  Subject: {sample_payload['subject']}")
print(f"  Routed to inbox: {inbox_id} ({route_key})")
print(f"  Crew result: [crew would run here with real API keys]")

✓ Production agent and crew defined
Agent role: Production Support Specialist
Agent tools: ['production_email_send']

Simulating webhook payload:
  Sender:  alice@example.com
  Subject: My dashboard keeps crashing
  Routed to inbox: i_sup001 (support)
  Crew result: [crew would run here with real API keys]


## Production Checklist

Before deploying a CrewAI email agent to production, verify:

| Pattern | What to check |
|---|---|
| Thread continuity | `ThreadRegistry` backed by persistent store (Redis/Postgres), not in-memory dict |
| Multi-inbox routing | One inbox per team/domain; inbox IDs in env vars, not hardcoded |
| Idempotency | Every `messages.send()` call goes through `send_with_retry()` |
| Dead-letter | Permanent failures (4xx) written to a queue that sends a Slack/email alert |
| Webhook signature | `commune.verify_signature()` called before any payload processing |
| Extraction schemas | Each inbox has a JSON schema so structured data arrives pre-parsed |

## Next Steps

- **[crewai_multi_agent_crew.ipynb](./crewai_multi_agent_crew.ipynb)** — multi-agent triage + QA pipeline
- **[openai_agents_02_patterns.ipynb](./openai_agents_02_patterns.ipynb)** — parallel agents, extraction schemas
- **[langchain_03_production.ipynb](./langchain_03_production.ipynb)** — deliverability monitoring, search, FastAPI deployment
- **Commune docs:** https://commune.email/docs